# Underwater Waste Detection — YOLO + SAM Segmentation

Two-stage pipeline:
1. **YOLOv8** detects bounding boxes (fast, accurate localization)
2. **SAM** (Segment Anything Model, ViT-H) refines each box into a pixel-accurate mask

This gives segmentation masks without training a segmentation model — SAM is used zero-shot,
prompted by the YOLO bounding boxes.

**Use case:** When you need to estimate trash volume/area rather than just count objects.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────
!nvidia-smi
!pip install -q ultralytics
!pip install -q 'git+https://github.com/facebookresearch/segment-anything.git'
!pip install -q supervision

In [ ]:
# ── Cell 2: Mount Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

DRIVE_BASE   = Path('/content/drive/MyDrive/underwater_waste_detection')
YOLO_DS_DIR  = DRIVE_BASE / 'trashcan_yolo'
WEIGHTS_DIR  = DRIVE_BASE / 'weights'
RESULTS_DIR  = DRIVE_BASE / 'yolo_sam_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/underwater-waste-detection')
if not REPO_DIR.exists():
    !git clone https://github.com/omprxkash/underwater-waste-detection.git {REPO_DIR}
sys.path.insert(0, str(REPO_DIR / 'src'))
os.chdir(REPO_DIR)

In [ ]:
# ── Cell 3: Download SAM weights ──────────────────────────────────────────
SAM_WEIGHTS_PATH = str(WEIGHTS_DIR / 'sam_vit_h.pth')
SAM_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'

if not Path(SAM_WEIGHTS_PATH).exists():
    print('Downloading SAM ViT-H weights (~2.5GB)...')
    !wget -q --show-progress "{SAM_URL}" -O "{SAM_WEIGHTS_PATH}"
    print('Download complete.')
else:
    print(f'SAM weights found: {SAM_WEIGHTS_PATH}')

In [ ]:
# ── Cell 4: Load YOLO + SAM models ───────────────────────────────────────
import torch
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamPredictor

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# YOLO — use best trained weights from notebook 02
YOLO_WEIGHTS = str(WEIGHTS_DIR / 'yolov8n_best.pt')
yolo_detector = YOLO(YOLO_WEIGHTS)
print(f'YOLO loaded: {YOLO_WEIGHTS}')

# SAM
sam = sam_model_registry['vit_h'](checkpoint=SAM_WEIGHTS_PATH)
sam.to(device=DEVICE)
sam_predictor = SamPredictor(sam)
print('SAM ViT-H loaded')

In [ ]:
# ── Cell 5: YOLO + SAM pipeline ───────────────────────────────────────────
import cv2
import numpy as np

CLASS_NAMES = ['trash', 'bio', 'rov']
MASK_COLORS = {
    'trash': np.array([220, 60,  60 ]),   # red
    'bio':   np.array([ 60, 180, 60 ]),   # green
    'rov':   np.array([ 60, 130, 220]),   # blue
}

def run_yolo_sam(image_bgr, yolo_model, sam_pred, conf=0.25, iou=0.45):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    # Stage 1: YOLO detection
    yolo_out = yolo_model(image_bgr, conf=conf, iou=iou, verbose=False)[0]
    if len(yolo_out.boxes) == 0:
        return [], [], [], []

    boxes  = yolo_out.boxes.xyxy.cpu().numpy()
    labels = yolo_out.boxes.cls.cpu().numpy().astype(int)
    scores = yolo_out.boxes.conf.cpu().numpy()

    # Stage 2: SAM segmentation using YOLO boxes as prompts
    sam_pred.set_image(image_rgb)
    all_masks = []
    for box in boxes:
        masks, seg_scores, _ = sam_pred.predict(box=box, multimask_output=True)
        best = masks[np.argmax(seg_scores)]
        all_masks.append(best)

    return all_masks, boxes, labels, scores

def visualize_yolo_sam(image_bgr, masks, boxes, labels, scores, alpha=0.45):
    annotated = image_bgr.copy().astype(np.float32)

    for mask, box, cls_idx, score in zip(masks, boxes, labels, scores):
        name  = CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else 'unk'
        color = MASK_COLORS.get(name, np.array([128, 128, 128]))
        color_bgr = color[::-1].tolist()

        # Overlay mask
        annotated[mask > 0] = (
            annotated[mask > 0] * (1 - alpha) + np.array(color_bgr) * alpha
        )

        # Draw box + label
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(annotated.astype(np.uint8), (x1, y1), (x2, y2), color_bgr, 2)
        text = f'{name} {score:.0%} | {int(mask.sum())}px'
        cv2.putText(annotated.astype(np.uint8), text, (x1+2, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_bgr, 2)

    return annotated.astype(np.uint8)

print('Pipeline functions defined.')

In [ ]:
# ── Cell 6: Run on sample images ──────────────────────────────────────────
import matplotlib.pyplot as plt
import random

val_images = list((YOLO_DS_DIR / 'images' / 'val').glob('*.jpg'))
sample_imgs = random.sample(val_images, min(4, len(val_images)))

fig, axes = plt.subplots(len(sample_imgs), 3, figsize=(18, 5*len(sample_imgs)))
if len(sample_imgs) == 1:
    axes = [axes]

for row_axes, img_path in zip(axes, sample_imgs):
    frame = cv2.imread(str(img_path))

    masks, boxes, labels, scores = run_yolo_sam(
        frame, yolo_detector, sam_predictor, conf=0.25
    )

    # Original
    row_axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    row_axes[0].set_title('Original', fontweight='bold')
    row_axes[0].axis('off')

    # YOLO only (boxes)
    yolo_only = frame.copy()
    for box, cls_idx, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = map(int, box)
        name = CLASS_NAMES[cls_idx] if cls_idx < 3 else 'unk'
        cv2.rectangle(yolo_only, (x1,y1),(x2,y2),(0,200,0),2)
        cv2.putText(yolo_only, f'{name} {score:.0%}', (x1+2, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,200,0), 2)
    row_axes[1].imshow(cv2.cvtColor(yolo_only, cv2.COLOR_BGR2RGB))
    row_axes[1].set_title(f'YOLO ({len(boxes)} detections)', fontweight='bold')
    row_axes[1].axis('off')

    # YOLO + SAM
    if masks:
        annotated = visualize_yolo_sam(frame, masks, boxes, labels, scores)
        row_axes[2].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        trash_area = sum(int(m.sum()) for m, l in zip(masks, labels) if l == 0)
        row_axes[2].set_title(f'YOLO + SAM | trash={trash_area}px', fontweight='bold')
    else:
        row_axes[2].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        row_axes[2].set_title('YOLO + SAM (no detections)', fontweight='bold')
    row_axes[2].axis('off')

plt.suptitle('YOLO + SAM Pipeline — Underwater Waste Segmentation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'yolo_sam_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/yolo_sam_comparison.png')

In [ ]:
# ── Cell 7: Trash coverage analysis ───────────────────────────────────────
# Estimate what fraction of each image is covered by detected trash

coverage_stats = []
analysis_images = val_images[:30]  # process 30 images

for img_path in analysis_images:
    frame = cv2.imread(str(img_path))
    h, w = frame.shape[:2]
    total_px = h * w

    masks, boxes, labels, scores = run_yolo_sam(frame, yolo_detector, sam_predictor, conf=0.3)

    trash_px = sum(int(m.sum()) for m, l in zip(masks, labels) if l == 0)
    bio_px   = sum(int(m.sum()) for m, l in zip(masks, labels) if l == 1)

    coverage_stats.append({
        'image': img_path.name,
        'n_detections': len(boxes),
        'n_trash': sum(1 for l in labels if l == 0),
        'trash_coverage_pct': trash_px / total_px * 100,
        'bio_coverage_pct': bio_px / total_px * 100,
    })

import pandas as pd
coverage_df = pd.DataFrame(coverage_stats)
print('Coverage Analysis (30 images):')
print(coverage_df.describe().round(2))

coverage_df.to_csv(str(RESULTS_DIR / 'coverage_analysis.csv'), index=False)
print(f'Saved: {RESULTS_DIR}/coverage_analysis.csv')